# Quick summary

In a multipumped system, the LLE can be written as: 

$$
\frac{\partial a (\theta, t)}{ \partial t} = \left(-1 + \delta \right)a + i\gamma |a|^2a + iF_0 + iF_- \mathrm{e}^{i\varpi t + i\mu_\mathrm{aux} \theta}
$$

If we assume that $\varpi>>0$ (and $|\varpi|<\omega_\mathrm{rep}/2$, obvious otherwise we jump in $\mu$). In this case, $\varpi$ denote the detuning the relative offset of the aux pump relative to the closest DKS comb tooth, of course if $\varpi$ becomes small enough, the nonlinearity becomes sufficient to lock it to zero and we enter the KIS regime (so the conidtion of $\varpi$ being large in our case). We also assume that the soliton indeed exists in the resonator exhibiting low dispersion such that the walk off time between pulses $\tau_\mathrm{wo}$ is smaller than the nonlinear time scale $\tau_\mathrm{nl}$, we can proceed to the mutli-color decomposition such that:

$$
a(\theta, t) = a_0(\theta, t) + a_\mathrm{-}(\theta, t) \mathrm{e}^{i\varpi t} + a_\mathrm{+}(\theta, t) \mathrm{e}^{-i\varpi t}
$$

Note that the \(\pm\) subscripts are arbitrary. Our convention is defined relative to \(\mu\), such that \(a_-\) has most of its energy at \(\mu < 0\), which in our case corresponds to the auxiliary comb, and we keep $a_0$ to be the soliton.

Reinserting this into the LLE, we obtain the coupled system.

$$
\begin{aligned}
\partial_t a_{-} (\theta, t) &= -a_{-} + i\gamma \left(|a_{-}|^2+2|a_{0}|^2+2|a_{+}|^2\right)a_{-} - i\gamma a_+^*a_0^2 + iF_\mathrm{e}^{\mu_\mathrm{aux}\theta}\\
\partial_t a_0 (\theta, t) &= \left(-1 + \delta \right)a_0 + i\gamma \left(2|a_{-}|^2+|a_{0}|^2+2|a_{+}|^2\right)a0  - 2i\gamma a_-a_+a_0^* + iF_0\\
\partial_t a_{+} (\theta, t) &= -a_{+} + i\gamma \left(2|a_{-}|^2+2|a_{0}|^2+2|a_{+}|^2\right)a_{+} - i\gamma a_-^*a_0^2\
\end{aligned}
$$

Now, note that there are obviously XPM terms, but also parametric ones that are essentially OPO. Here it is not a CW OPO but a pulsed OPO, which tells you at which phase they exist (i.e., $n\varpi$). In the three-color case, consider $a_+$ at $-\varpi$, but you can extend this to other colors ($a_{2+}, a_{2-}$, etc.) and account for the nonlinear interaction accordingly. The mode at which this happens is simply set by the phase-matching condition, i.e., where the color $a_X$ is resonant (or closest to resonance). I am still sure there is some weird PT-symmetry physics to explore between the phase-matched and non-phase-matched cases. The energy flow also comes from the complex-conjugate term: in $a_+$, since we drive with $a_-^*$, energy transfer toward $-\mu_\mathrm{aux}$ is more efficient.

(Note that the parametric term in \(a_0\) is the one at the origin of the PDCS.)

(might be some typo on sign and all, so refer to this [paper](https://opg.optica.org/oe/fulltext.cfm?uri=oe-33-10-21824), this [paper](https://www.nature.com/articles/s41566-024-01540-w), and this [one](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.134.193802) to actually check the math)

# Function that processes the data

This function essentially finds the repetition rate between two maximum comb teeth (using the threshold you specify) and divides it by the number of comb teeth between them. It does not work perfectly, which is why it is possible to fit the “slope” between selected indices to obtain a more accurate repetition rate.

It then slices the spectrum into windows of $[-f_\mathrm{rep}/2, f_\mathrm{rep}/2]$, centered around $N \times f_\mathrm{rep} + f_0$, where $f_0$ is the strongest comb tooth (usually the pump), and creates a 2D spectrum: x-axis = $\mu$; y-axis = frequency (or phase, if divided by $f_\mathrm{rep}$).

This makes it possible to retrieve the spectrogram and its different colors.

Additional processing can extract only the peaks of the different colors to make them easier to observe.

In [1]:
## imports 
import numpy as np
import pandas as pd
from scipy import signal
import plotly.graph_objects as go
from cmcrameri import cm as cmap
import plotly.io as pio
pio.templates.default = "simple_white"
def mpl_to_plotly(cmap, pl_entries=255, rdigits=15):
    """Convert a Matplotlib colormap into a Plotly colorscale."""
    scale = np.linspace(0, 1, pl_entries)
    colors = [
        [int(channel) for channel in color]
        for color in cmap(scale)[:, :3] * 255
    ]
    return [
        [float(round(value, rdigits)), f"rgb{tuple(color)}"]
        for value, color in zip(scale, colors)
    ]

In [2]:

def getMap(df, FSR=1.0e12, Mtot=[-110, 120], f0=289.6569e12, res=1000, window=80e9, pks_only = False):
    omega = np.linspace(-FSR / 2, FSR / 2, res)
    data = pd.DataFrame(index=omega, columns=np.arange(Mtot[0], Mtot[1] + 0.1, 1))
    ipmp = np.abs(df.freq - f0).idxmin()
    fpmp = df.freq[ipmp]
    for ii in range(Mtot[0], Mtot[1] + 1):
        mask = (df.freq > fpmp + ii * FSR - FSR / 2) & (
            df.freq < fpmp + ii * FSR + FSR / 2
        )
        freq = df.freq[mask] - fpmp - ii * FSR
        if df.S[mask].size > 0:
            S = np.interp(omega, freq, df.S[mask])

            if pks_only:
                pks, _ = signal.find_peaks(S, distance=10)
                data[ii] = -100.0
                data.loc[data.index[pks], ii] = S[pks]
            else:
                data[ii] = S
    mask = np.abs(data.index.values) < window
    data = data[mask]
    return data


def GetOSAspectro(
    osa,
    do_refit=[],
    FSR=None,
    FSR0=1e12,
    height=-12,
    distance=100,
    prominence=10,
    res=900,
    window=30e9,
    freq_shift = None,
    Mtot = [-110, 120], 
    pks_only = False, 
    ceo_estimate = False,
    N0 = None,
    pdcs = True,
):

    pks_pmp, _ = signal.find_peaks(osa.S, distance=100, height=height, prominence=prominence)
    FSR0 = 1e12
    fdiff = osa.iloc[pks_pmp].freq.diff().values[1]
    N = np.round(fdiff / FSR0)
    if not FSR:
        if N0: 
            FSR = osa.iloc[pks_pmp].freq.diff().values[1] / N0
            N = N0
        else: 
            FSR = osa.iloc[pks_pmp].freq.diff().values[1] / N

    if pdcs: 
        f0 = osa.iloc[pks_pmp].freq.sum() / 2
        i0 = np.abs(osa.freq - f0).idxmin()
        f0 = f0 + np.mod(N, 2) * FSR / 2
    else: 
        f0 = osa.iloc[pks_pmp].sort_values(by='S', ascending=False).freq.values[0]
        i0 = np.abs(osa.freq - f0).idxmin()
    
    osa["mu"] = (osa.freq - f0) / FSR
    
    data = getMap(
        osa, FSR=FSR, Mtot=Mtot, f0=f0, res=res, window=window, pks_only = pks_only
    )

    if do_refit: 
        if len(do_refit) == 2:
            mu = data.columns.values.astype(float)
            freq = data.index.values.astype(float)
            S = data.values.astype(float)
            mask_mu = np.logical_and(mu >= do_refit[0], mu <= do_refit[1])
            ridge = freq[np.argmax(S, axis=0)]
            m, b = np.polyfit(mu[mask_mu], ridge[mask_mu], deg=1)
            FSR = FSR + m
            data = getMap(
                osa,
                FSR=FSR,
                Mtot=Mtot,
                f0=f0 + np.mod(N, 2) * FSR / 2,
                res=res,
                window=window,
                pks_only = pks_only
            )
            if freq_shift is None: 
                data.index -= data[0].idxmax() 
            else:
                data.index -= freq_shift
    if ceo_estimate:
        ceo = 2*osa.freq[pks_pmp[0]] - osa.freq[pks_pmp[1]]
        return data, FSR, f0, N, ceo
    return data, FSR, f0, N, _

# Processing

First, we load the data, that's the one from this [paper](https://pubs.aip.org/aip/app/article/11/3/030801/3381796)

There is a DKS, a cooler at around 307 THz, and an auxiliary pump at 193 THz, all passing through the DW. The key point is that there is a hidden third color in the 340 THz band. From the spectrum, this is not immediately clear, right?

In [3]:
trace = pd.read_csv('./Spectra_pzt40.50_stitched.csv')
fig = go.Figure()
tr = [go.Scatter(x = trace.freq*1e-12, y = trace.S, showlegend=False)]
fig.add_traces(tr)
fig.update_xaxes(title = 'Frequency (THz)')
fig.update_yaxes(title = 'Power (dBm)')
fig.show()

Usually it is better to see it in mode number for the spectrogram, so here I process the spectrum into mode number, with the pump being µ = 0

In [4]:
trace['mu'] = trace.freq - trace.freq[trace.S.idxmax()]
trace['mu'] /= 997e9

fig = go.Figure()
tr = [go.Scatter(x = trace.mu, y = trace.S, showlegend=False)]
fig.add_traces(tr)
fig.show()

Then we process the data. First let's just process it into a spectrogram without much else.

In [5]:
data, FSR, f0, N, fceo = GetOSAspectro(
    trace,
    FSR0=1000e9, #rough estimate in our case
    Mtot=[-107, 107],
    height=7, # what are the two pump above
    res=900, # resolution of the spectrogram
    window=498.5e9, #here the full, that is about frep/2
    ceo_estimate=True,
    pdcs=False, # here a switch since it works also for pdcs
)

print(f"νrep = {FSR*1e-9} GHz")
Nmod = np.round(fceo / FSR)
fceo_ = fceo - Nmod * FSR
print(f"νceo: {fceo_*1e-9} GHz")

fig = go.Figure()
tr = go.Heatmap(
    x=data.columns,
    y=data.index * 1e-9,
    z=data.values,
    colorscale=mpl_to_plotly(cmap.oslo_r),
    zsmooth="best",
    colorbar=dict(title="Power [dBm]"),
    zmax=0,
    zmin=-80,
)
fig.add_trace(tr)
fig.update_xaxes(title = 'mode number µ')
fig.update_yaxes(title = 'Frequency [GHz]')
fig.show()

νrep = 997.586067985839 GHz
νceo: 56.44665281042188 GHz


We can clearly see the DKS and the cooler. We can infer that something is happening with the auxiliary pump, and perhaps something else as well. However, the other color on the other side is not very clear. We can zoom a bit first, since the cooler is not doing much actually (it is actually doing some weird bragg scattering at µ = -70, that is mediated by the main DKS and the auxiliary pump, we can discuss that further if you want)

In [6]:
data, FSR, f0, N, fceo = GetOSAspectro(
    trace,
    FSR0=1000e9,
    Mtot=[-107, 107],
    height=7,
    res=900,
    window=60e9,  ## Reduce that to see it better
    pks_only=False,
    ceo_estimate=True,
    pdcs=False,
)

print(f"νrep = {FSR*1e-9} GHz")
Nmod = np.round(fceo / FSR)
fceo_ = fceo - Nmod * FSR
print(f"νceo: {fceo_*1e-9} GHz")

fig = go.Figure()
tr = go.Heatmap(
    x=data.columns,
    y=data.index * 1e-9,
    z=data.values,
    colorscale=mpl_to_plotly(cmap.oslo_r),
    zsmooth="best",
    colorbar=dict(title="Power [dBm]"),
    zmax=0,
    zmin=-80,
)
fig.add_trace(tr)
fig.update_xaxes(title = 'mode number µ')
fig.update_yaxes(title = 'Frequency [GHz]')
fig.show()

νrep = 997.586067985839 GHz
νceo: 56.44665281042188 GHz


Ok that's better, but clearly we don't see things well still, especially at high mode number (high frequency) becasue of the OSA resolution. Let's only process the max of the spectrogram: 


In [7]:
data, FSR, f0, N, fceo = GetOSAspectro(
    trace,
    FSR0=1000e9,
    Mtot=[-107, 107],
    height=7,
    res=900,
    window=60e9, 
    pks_only=True, #Now trigger that 
    ceo_estimate=True,
    pdcs=False,
)

print(f"νrep = {FSR*1e-9} GHz")
Nmod = np.round(fceo / FSR)
fceo_ = fceo - Nmod * FSR
print(f"νceo: {fceo_*1e-9} GHz")

fig = go.Figure()
tr = go.Heatmap(
    x=data.columns,
    y=data.index * 1e-9,
    z=data.values,
    colorscale=mpl_to_plotly(cmap.oslo_r),
    zsmooth="best",
    colorbar=dict(title="Power [dBm]"),
    zmax=0,
    zmin=-90,
)
fig.add_trace(tr)
fig.update_xaxes(title = 'mode number µ')
fig.update_yaxes(title = 'Frequency [GHz]')
fig.show()

νrep = 997.586067985839 GHz
νceo: 56.44665281042188 GHz


OK much better, but our estimate of the FSR is not accurate. We can fit the slope of the DKS, since it should be zero all the way (i.e. the referefernce of our freuqency marker grid). 

In [8]:
data, FSR, f0, N, fceo = GetOSAspectro(
    trace,
    FSR0=1000e9,
    Mtot=[-107, 107],
    do_refit=[-68, 45], # Refit the slope between this modes
    height=7,
    res=900,
    window=60e9, 
    pks_only=True, #Now trigger that 
    ceo_estimate=True,
    pdcs=False,
)

print(f"νrep = {FSR*1e-9} GHz")
Nmod = np.round(fceo / FSR)
fceo_ = fceo - Nmod * FSR
print(f"νceo: {fceo_*1e-9} GHz")

fig = go.Figure()
tr = go.Heatmap(
    x=data.columns,
    y=data.index * 1e-9,
    z=data.values,
    colorscale=mpl_to_plotly(cmap.oslo_r),
    zsmooth="best",
    colorbar=dict(title="Power [dBm]"),
    zmax=0,
    zmin=-90,
)
fig.add_trace(tr)
fig.update_xaxes(title = 'mode number µ')
fig.update_yaxes(title = 'Frequency [GHz]')
fig.show()

νrep = 997.2273834377639 GHz
νceo: 93.03247671407813 GHz


Alright, now it is much better! Note that the repetition rate has changed; we now have a much more accurate value. FYI, this is how we estimate the repetition rate used to drive the EO comb, which lets us perform all the metrology by frequency-down-converting the high repetition rate into a 50 MHz window.

If you look at it, you can clealry see that the aux comb is offset by about -33 GHz from the DKS, and that the new color, that appear from teh nonlinear mixung appears on the other side at about +33 GHz, as expected

----

Of course, we can do it on a mutlicolor soliton that is much more obvious, the above example was really to show that it enables to observe things that are not necessarly obivous

In [9]:
trace = pd.read_csv("./Spectra_pzt52.50_stitched.csv")
fig = go.Figure().set_subplots(rows = 2, cols = 1, shared_xaxes=True, vertical_spacing=0.02)
trace['mu'] = trace.freq - trace.freq[trace.S.idxmax()]
trace['mu'] /= 997e9
tr = [go.Scatter(x = trace.mu, y = trace.S, showlegend=False)]

data, FSR, f0, N, fceo = GetOSAspectro(
    trace,
    FSR0=1000e9,
    Mtot=[-107, 107],
    do_refit=[-68, 45], # Refit the slope between this modes
    height=7,
    res=900,
    window=60e9, 
    pks_only=True, #Now trigger that 
    ceo_estimate=True,
    pdcs=False,
)

print(f"νrep = {FSR*1e-9} GHz")
Nmod = np.round(fceo / FSR)
fceo_ = fceo - Nmod * FSR
print(f"νceo: {fceo_*1e-9} GHz")

tr += [go.Heatmap(
    x=data.columns,
    y=data.index * 1e-9,
    z=data.values,
    colorscale=mpl_to_plotly(cmap.oslo_r),
    zsmooth="best",
    colorbar=dict(title="Power [dBm]"),
    zmax=0,
    zmin=-90,
)]
fig.add_traces(tr, rows = [1,2], cols = [1,1])
fig.update_xaxes(title = 'mode number µ', row = 2)
fig.update_yaxes(title = 'Power [dB]', row = 1)
fig.update_yaxes(title = 'Frequency [GHz]', row = 2)
fig.update_layout(height=800)
fig.show()


νrep = 997.2499654460743 GHz
νceo: 105.4305770789375 GHz
